# MapleStory Idle AOS — D7 ROAS Weekly Trend

| | |
|---|---|
| **Bundle** | `com.nexon.ma` |
| **OS** | Android |
| **Period** | Last 90 days (rolling from run date) |
| **Primary KPI** | D7 ROAS · Target: **39%** (source: Nexon product one-pager) |
| **Table** | `moloco-ae-view.market_share.fact_market` |
| **Author** | Haewon Yum · KOR GDS |
| **Created** | 2026-04-29 |

In [ ]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    print(f'✅ {label}: {len(df)} rows')
    return df

BUNDLE   = 'com.nexon.ma'
OS       = 'ANDROID'
D7_TARGET = 39.0  # Nexon-stated D7 ROAS target (%)

## Section 1 — Overall Weekly D7 ROAS

In [ ]:
Q_OVERALL = """
SELECT
  DATE_TRUNC(install_date_utc, WEEK(MONDAY))                             AS week_start,
  SUM(moloco.gross_spend_usd)                                            AS spend_usd,
  SUM(moloco.installs)                                                   AS installs,
  SAFE_DIVIDE(SUM(moloco.kpi_revenue_d1), SUM(moloco.gross_spend_usd))   AS d1_roas,
  SAFE_DIVIDE(SUM(moloco.kpi_revenue_d7), SUM(moloco.gross_spend_usd))   AS d7_roas
FROM `moloco-ae-view.market_share.fact_market`
WHERE
  app_market_bundle = '{bundle}'
  AND os             = '{os}'
  AND install_date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
                           AND CURRENT_DATE()
GROUP BY 1
ORDER BY 1
""".format(bundle=BUNDLE, os=OS)

df_overall = run_query(Q_OVERALL, 'Overall weekly')
df_overall['week_start']   = pd.to_datetime(df_overall['week_start'])
df_overall['d7_roas_pct']  = df_overall['d7_roas'] * 100
df_overall['d1_roas_pct']  = df_overall['d1_roas'] * 100
df_overall

## Section 2 — Weekly D7 ROAS by Geo

In [ ]:
Q_GEO = """
WITH base AS (
  SELECT
    DATE_TRUNC(install_date_utc, WEEK(MONDAY)) AS week_start,
    CASE
      WHEN country IN ('KOR','TWN','USA','THA','CAN','SGP','MYS') THEN country
      ELSE 'Other'
    END AS geo,
    SUM(moloco.kpi_revenue_d7)  AS rev_d7,
    SUM(moloco.gross_spend_usd) AS spend_usd
  FROM `moloco-ae-view.market_share.fact_market`
  WHERE
    app_market_bundle = '{bundle}'
    AND os = '{os}'
    AND install_date_utc BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY)
                             AND CURRENT_DATE()
  GROUP BY 1, 2
)
SELECT
  week_start,
  geo,
  SAFE_DIVIDE(rev_d7, spend_usd) AS d7_roas,
  spend_usd
FROM base
ORDER BY 1, 2
""".format(bundle=BUNDLE, os=OS)

df_geo = run_query(Q_GEO, 'Geo weekly')
df_geo['week_start']  = pd.to_datetime(df_geo['week_start'])
df_geo['d7_roas_pct'] = df_geo['d7_roas'] * 100

# Pivot to wide format
df_pivot = df_geo.pivot_table(
    index='week_start', columns='geo', values='d7_roas_pct', aggfunc='first'
)
df_pivot.round(1)

## Section 3 — Charts

In [ ]:
# ── Chart 1: Overall D7 ROAS weekly trend ──────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

# Separate complete vs partial weeks (last week is partial — D7 cohort not matured)
df_complete = df_overall.iloc[:-1]
df_partial  = df_overall.iloc[-1:]

ax.plot(df_complete['week_start'], df_complete['d7_roas_pct'],
        color='#4f46e5', linewidth=2.5, marker='o', markersize=5, label='D7 ROAS')
ax.fill_between(df_complete['week_start'], df_complete['d7_roas_pct'],
                alpha=0.10, color='#4f46e5')

# Partial week (dashed)
ax.plot([df_complete['week_start'].iloc[-1], df_partial['week_start'].iloc[0]],
        [df_complete['d7_roas_pct'].iloc[-1], df_partial['d7_roas_pct'].iloc[0]],
        color='#4f46e5', linewidth=1.5, linestyle=':', marker='o', markersize=4)
ax.annotate('partial\n(D7 pending)', xy=(df_partial['week_start'].iloc[0],
            df_partial['d7_roas_pct'].iloc[0]),
            xytext=(8, 8), textcoords='offset points', fontsize=8, color='#9ca3af')

# Target line
ax.axhline(D7_TARGET, color='#ef4444', linestyle='--', linewidth=1.5,
           label=f'Nexon target ({D7_TARGET:.0f}%)')
ax.text(df_overall['week_start'].min(), D7_TARGET + 1.5, 'Target 39%',
        color='#ef4444', fontsize=8)

ax.set_title('MapleStory Idle (AOS) — Overall Weekly D7 ROAS', fontsize=14,
             fontweight='bold', pad=12)
ax.set_ylabel('D7 ROAS (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_facecolor('#f9fafb')
fig.patch.set_facecolor('white')
fig.tight_layout()
plt.savefig('overall_d7roas_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: overall_d7roas_trend.png')

In [ ]:
# ── Chart 2: D7 ROAS by geo (weekly) ──────────────────────────────────────
GEO_COLORS = {
    'KOR': '#4f46e5',
    'USA': '#f59e0b',
    'THA': '#10b981',
    'TWN': '#ef4444',
    'SGP': '#8b5cf6',
    'MYS': '#06b6d4',
}
FOCUS_GEOS = ['KOR', 'USA', 'THA', 'TWN']  # highest spend + most signal

fig, ax = plt.subplots(figsize=(13, 5))

for geo in FOCUS_GEOS:
    if geo not in df_pivot.columns:
        continue
    series = df_pivot[geo].dropna()
    ax.plot(series.index, series.values,
            color=GEO_COLORS.get(geo, '#6b7280'),
            linewidth=2, marker='o', markersize=4, label=geo)

# Target line
ax.axhline(D7_TARGET, color='#374151', linestyle='--', linewidth=1.2,
           alpha=0.6, label=f'Target ({D7_TARGET:.0f}%)')

ax.set_title('MapleStory Idle (AOS) — Weekly D7 ROAS by Geo', fontsize=14,
             fontweight='bold', pad=12)
ax.set_ylabel('D7 ROAS (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30, ha='right')
ax.legend(fontsize=10, loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_facecolor('#f9fafb')
fig.patch.set_facecolor('white')
fig.tight_layout()
plt.savefig('geo_d7roas_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: geo_d7roas_trend.png')

## Key Observations

**Target: D7 ROAS 39%** (Nexon-stated; source: MapleStory_Idle_AOS_GL_Product_Team_OnePager)

**1. No geo sustains target-level ROAS.**
The Jan 26 week (launch cohort) was the only time KOR approached target (42%). USA briefly spiked to 78% the week of Feb 23 — likely a small high-LTV test batch. Both declined immediately after.

**2. KOR (72% of spend) runs at ~10–16% D7 ROAS in steady state** — roughly 1/3 of the 39% target. CPI simultaneously rose from $17 at launch to $30+ now: classic audience exhaustion in the primary geo.

**3. USA is the most efficient geo** (28.3% 90-day D7 ROAS, $15.56 CPI) but dropped off after February. At $44K total vs. $997K in KOR, it is significantly underleveraged.

**4. THA shows healthy unit economics** ($5.89 CPI, 16.3% D7 ROAS) and more stability than USA. Limited scale ($12K total spend) but the quality signal is real.

**5. TWN near-zero ROAS throughout** — $153K in spend with 1.4% D7 ROAS. CAN and AUS similarly at <1%. These geos warrant a continuation review.

**6. The Apr 27 data point is partial** (cohort not yet at D7) — exclude from any snapshot comparisons.